# **1- Install & Import libraries**

In [ ]:
# Install all required libraries for NLP model training and fine-tuning
# - transformers: for loading and using large language models
# - accelerate: for optimized training on CPU/GPU
# - peft: for lightweight fine-tuning (LoRA)
# - bitsandbytes: for 4-bit/8-bit model quantization
# - datasets: for loading and processing datasets easily

!pip install -U transformers accelerate peft bitsandbytes datasets

In [ ]:
import torch # Load the PyTorch library
# Check if a GPU (CUDA) is available
if torch.cuda.is_available():
  # Print the name of the available GPU device
  print(torch.cuda.get_device_name(0))

Tesla T4


In [ ]:
import pandas as pd
from peft import PeftModel # For loading models trained with LoRA (PEFT)
from datasets import Dataset # HuggingFace dataset structure
from transformers import (
    AutoTokenizer, # Converts text into tokens the model can understand
    AutoModelForCausalLM, # Loads a pre-trained causal language model (LLM)
    BitsAndBytesConfig, # Enables 4-bit / 8-bit model quantization
    Trainer, # High-level training loop for fine-tuning
    TrainingArguments,
    DataCollatorForLanguageModeling,
)
# Import PEFT and LoRA utilities
from peft import(
    LoraConfig, # LoRA configuration settings
    get_peft_model, # Apply LoRA layers to the base model
    prepare_model_for_kbit_training,  # Prepare model for 4-bit or 8-bit training
)

# **2- Load and Clean stories data**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Path to the CSV dataset stored in Google Drive
path = "/content/drive/MyDrive/NLP/NLP_V2_Final/ALLaM_Fine_tune/Formatted-MSA-prompts-stories-for-fine-tuning.csv"
df = pd.read_csv(path)

# Remove rows where either Prompt or Story is missing (NaN)
df = df.dropna(subset = ["Prompt", "Story"])

# Remove the unwanted auto-generated column "Unnamed: 0"
df = df.drop(["Unnamed: 0"], axis = 1)

# Print the number of rows after cleaning
print("Rows:", len(df))
df.head()

Rows: 996


,Prompt,Story
0,أكتب قصة قصيرة تكون مفهومة للأطفال في عمر 5-8....,في قصرٍ عظيم في قطر، كان يعيش الأمير جمال وقطه...
1,أكتب قصة قصيرة تكون مفهومة للأطفال في عمر 3-5....,في قلعة كبيرة وعتيقة في جيبوتي، كان يعيش فارس ...
2,أكتب قصة قصيرة تكون مفهومة للأطفال في عمر 3-5....,في يوم مشمس جميل في الكويت، كانت هناك بطة صغير...
3,أكتب قصة قصيرة تكون مفهومة للأطفال في عمر 3-5....,في بلد جميل يدعى فلسطين، كان يعيش صبي صغير يدع...
4,أكتب قصة قصيرة تكون مفهومة للأطفال في عمر 8-10...,في قرية صغيرة في السودان، بينما يلف الشتاء ببر...


# **3- Build HuggingFace Dataset**

In [ ]:
# Convert the pandas DataFrame into a HuggingFace Dataset
dataset = Dataset.from_pandas(df)
len(dataset), dataset[0]

(996,
 {'Prompt': 'أكتب قصة قصيرة تكون مفهومة للأطفال في عمر 5-8. أحداث القصة يجب أن تدور في القصر. نهاية القصة يجب أن تكون سعيدة. يجب أن يكون هناك حوار. عدد الشخصيات في القصة يجب أن يكون 2. بلد الأحداث هي  قطر. الأحداث تدور في فصل الصيف. يجب أن تحتوي القصة على شعور بالانزعاج.  أكتب القصة مباشرة.',
  'Story': 'في قصرٍ عظيم في قطر، كان يعيش الأمير جمال وقطه الأليف ميسو. كان الصيف في كامل ألقه، لكن الحرارة كانت شديدة لدرجة أن جمال وميسو شعرا بالانزعاج الشديد. \n\nذات يوم، قال جمال متأففًا: "يا ميسو، كيف يمكننا الاستمتاع بهذا الصيف والحرارة شديدة هكذا؟"\n\nميسو، بنظرة حزينة رد مواءً كأنه يوافق صديقه الأمير على شعوره بالانزعاج.\n\nفكر الأمير جمال قليلاً وقال: "لدي فكرة! ما رأيك أن نحول القصر إلى مكان للمتعة واللعب؟ يمكننا إقامة مسبح مؤقت في حديقة القصر."\n\nلمعت عينا ميسو بفرح، وبدا وكأنه يبتسم موافقًا على الفكرة.\n\nبسرعة، عمل الأمير جمال وقطه ميسو على تجهيز المسبح. كان جمال يضحك وميسو يطارد الفقاعات المائية ويحاول القفز للإمساك بها. تحول القصر من مكان يسوده الانزعاج بسبب الحر إلى مساحة م

# **4- Load ALLam in 4-bit + Apply LoRA Configuration**

In [ ]:
# Name of the pre-trained ALLaM model to download from HuggingFace
MODEL_NAME = "humain-ai/ALLaM-7B-Instruct-preview"

# Directory where the fine-tuned LoRA adapter and training outputs will be saved
OUTPUT_DIR = "/content/drive/MyDrive/NLP/NLP_V2_Final/ALLaM_Fine_tune/allam-arastories-lora"

In [ ]:
## Load model in 4-bit quantization:
# Reduces VRAM usage, speeds up loading, enables fine-tuning on small GPUs,
# While keeping model quality very close to full precision.

bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_compute_dtype = torch.float16,
)

In [ ]:
# Loads the tokenizer associated with the model(ALLaM).
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME,trust_remote_code=True)

# Checks if the model lacks a padding token.
if tokenizer.pad_token_id is None:
  # If not, use the EOS (end-of-sequence) token as the padding token
    tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

In [ ]:
# Load the ALLaM model with 4-bit quantization enabled
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config = bnb_config, # Apply 4-bit bitsandbytes configuration
    trust_remote_code=True # Allow custom model code from the repository
    )

config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.03G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [ ]:
# Prepare 4-bit quantized model for training
model = prepare_model_for_kbit_training(model)

# Disable caching during training to avoid memory issues
model.config.use_cache = False

# Enable gradient checkpointing to reduce GPU memory usage during training
model.gradient_checkpointing_enable()


In [ ]:
# Configure LoRA (Low-Rank Adaptation) settings for efficient fine-tuning
peft_config = LoraConfig(
    r=32,        # Rank of LoRA matrices (higher = more capacity)
    lora_alpha=16, # Scaling factor for LoRA updates
    lora_dropout=0.1, # Dropout to prevent overfitting
    bias="none", # Do not train any bias parameters
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], # Apply LoRA to attention layers
)

In [ ]:
# Apply the LoRA configuration to the base model
model = get_peft_model(model, peft_config)

# Print how many parameters are trainable vs. frozen
model.print_trainable_parameters()

trainable params: 33,554,432 || all params: 7,034,114,048 || trainable%: 0.4770


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("Model on:", next(model.parameters()).device)

Model on: cuda:0


# **5- Prepare Tokenization and Chat Formatting**

In [ ]:
def build_text(prompt, story):
    # Prepare the conversation structure for ALLaM in chat format
    messages = [
        {
            "role": "system",
            "content": "أنت علام، نموذج مخصص لكتابة قصص عربية ممتعة ومناسبة للأطفال بناءً على طلب المستخدم."
        },
        {"role": "user", "content": prompt}, # User's request
        {"role": "assistant", "content": story},  # Correct story (label for training)
    ]

    # Convert messages into ALLaM chat template format (no tokenization yet)
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return text # Return the formatted chat text

In [ ]:
# Maximum sequence length for tokenized inputs
MAX_SEQ_LEN = 512

# Build chat-formatted text for each (prompt, story) pair
def tokenize_batch(examples):
    texts = [build_text(p, s) for p, s in zip(examples["Prompt"], examples["Story"])]

    # Tokenize the texts with truncation and fixed-length padding
    enc = tokenizer(
        texts,
        max_length=MAX_SEQ_LEN,
        truncation=True,
        padding="max_length",
    )

    # Prepare labels: ignore padding tokens by replacing them with -100
    labels = []
    pad_id = tokenizer.pad_token_id
    for ids in enc["input_ids"]:
        labels.append([tok if tok != pad_id else -100 for tok in ids])

    # Add labels to the tokenized batch
    enc["labels"] = labels
    return enc

In [ ]:
# Apply tokenization to the entire dataset using the tokenize_batch function
tokenized_dataset = dataset.map(
    tokenize_batch, # Function that tokenizes each batch
    batched=True, # Process multiple samples at once (faster)
    remove_columns=["Prompt", "Story"], # Remove original text columns after tokenizing
)

# Display the keys of the first sample and the total dataset size
tokenized_dataset[0].keys(), len(tokenized_dataset)

Map:   0%|          | 0/996 [00:00<?, ? examples/s]

(dict_keys(['input_ids', 'attention_mask', 'labels']), 996)

# **6- Create a Smaller Subset for Light Training**

In [ ]:
# Number of samples to use for lightweight training
# 300 = very light, 600–1000 = normal
N_SAMPLES = 650

# Select a smaller subset of the dataset for faster training
small_dataset = tokenized_dataset.select(range(min(N_SAMPLES, len(tokenized_dataset))))  # Ensure we don't exceed dataset size


# Print the number of samples actually selected
len(small_dataset)


650

# **7- Configure Training Arguments**

In [ ]:
# Training configuration for fine-tuning the LoRA adapter
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,              # Where to save checkpoints and final model
    num_train_epochs=2,                 # Number of full passes over the dataset
    per_device_train_batch_size=1,      # Small batch size to fit into GPU memory
    gradient_accumulation_steps=4,      # Effective batch size = 1 × 4 = 4
    learning_rate=4e-5,                 # Learning rate for LoRA training
    logging_steps=20,                   # Log training progress every 20 steps
    save_steps=200,                     # Save a checkpoint every 200 steps
    save_total_limit=2,                 # Keep only the last 2 checkpoints
    fp16=True,                          # Enable mixed precision for faster training
    report_to="none",                   # Disable logging to external services
)

# **8- Fine-Tune ALLaM Using LoRA (Training Loop)**

In [ ]:
# Data collator prepares batches for causal language modeling (GPT-style)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False, # Disable masked language modeling (not used for causal LM)
)

# Initialize the Trainer with model, training args, dataset, and collator
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_dataset,
    data_collator=data_collator,
)

# Start fine-tuning the LoRA adapter
trainer.train()

# Save the fine-tuned model (LoRA weights) and tokenizer
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)


Step,Training Loss
20,2.337900
40,1.834300
60,1.659000
80,1.614300
100,1.566100
120,1.529800
140,1.552600
160,1.545400
180,1.544400
200,1.532100


('/content/drive/MyDrive/NLP/NLP_V2_Final/ALLaM_Fine_tune/allam-arastories-lora/tokenizer_config.json',
 '/content/drive/MyDrive/NLP/NLP_V2_Final/ALLaM_Fine_tune/allam-arastories-lora/special_tokens_map.json',
 '/content/drive/MyDrive/NLP/NLP_V2_Final/ALLaM_Fine_tune/allam-arastories-lora/chat_template.jinja',
 '/content/drive/MyDrive/NLP/NLP_V2_Final/ALLaM_Fine_tune/allam-arastories-lora/tokenizer.model',
 '/content/drive/MyDrive/NLP/NLP_V2_Final/ALLaM_Fine_tune/allam-arastories-lora/added_tokens.json',
 '/content/drive/MyDrive/NLP/NLP_V2_Final/ALLaM_Fine_tune/allam-arastories-lora/tokenizer.json')

# **9- Test the Fine-Tuned ALLaM Model**

In [ ]:
# Load the base ALLaM model in 4-bit quantization
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    trust_remote_code=True,
).to(device)

# Load the fine-tuned LoRA adapter on top of the base model
ft_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR).to(device)

# Set the model to evaluation mode
ft_model.eval()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(64000, 4096)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.

In [ ]:
test_messages = [
    {
        "role": "system",
        "content": "أنت علام، كاتب قصص عربية للأطفال يركز على القيم الإسلامية والعبرة في النهاية."
    },
    {
        "role": "user",
        "content": "اكتب قصة قصيرة لطفلة عمرها ٨ سنوات عن قيمة الصبر، تدور في جدة وتنتهي بعبرة واضحة."
    },
]

# Convert the test messages into ALLaM chat format
# add_generation_prompt=True: adds an <assistant> tag so the model knows to generate a response
chat_text = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False, # Return raw text, not tokenized yet
    add_generation_prompt=True, # Prepare for text generation
)

In [ ]:
# Tokenize the chat-formatted text and move it to the device (GPU/CPU)
inputs = tokenizer(chat_text, return_tensors="pt").to(device)

# Generate text without computing gradients (inference mode)
with torch.no_grad():
    out = ft_model.generate(
        **inputs,
        max_new_tokens=400,   # Maximum length of the generated story
        do_sample=True,       # Enable sampling for creative outputs
        top_p=0.9,            # Nucleus sampling (controls diversity)
        temperature=0.8,      # Soft randomness for story creativity
    )

# Decode the tokens back into readable text
print(tokenizer.decode(out[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[INST] <<SYS>>
أنت علام، كاتب قصص عربية للأطفال يركز على القيم الإسلامية والعبرة في النهاية.
<</SYS>>

اكتب قصة قصيرة لطفلة عمرها ٨ سنوات عن قيمة الصبر، تدور في جدة وتنتهي بعبرة واضحة. [/INST] في مدينة جدة الجميلة، حيث يجتمع البحر الأزرق والسماء الزرقاء، كانت هناك طفلة صغيرة تدعى ليلى. كانت ليلى تحب اللعب مع أصدقائها في الحديقة الواسعة أمام منزلها، ولكنها كانت دائماً تشعر بالضيق عندما لا تجد شيئاً تفعله في وقت الفراغ.

ذات يوم، قررت ليلى أن تتعلم الصبر. قالت لنفسها إن الصبر هو مفتاح النجاح. لذا، قررت أن تنتظر بصبر وصول أصدقائها للعب معها.

مر الوقت ببطء شديد، ولم يأتِ أصدقاؤها بعد. بدأت ليلى تشعر بالحزن لأنها ظنت أنها ستقضي وقتاً طويلاً وحدها. لكنها تذكرت قرارها وقررت أن تنتظر بصبر.

بعد وقت طويل، سمعت ليلى صوت سيارة أصدقائها تقترب. فتحت الباب واستقبلتهم بفرحة كبيرة، وقد تعلمت درساً قيماً: الصبر يجلب الراحة والسعادة.

وهكذا، بينما كانت تلعب مع أصدقائها، أدركت أن الصبر ليس فقط مفتاحاً للنجاح، ولكنه أيضاً يجعل الحياة أكثر حلاوة.

وعلى الرغم من أنها كانت تشعر بالضيق في البداية، إلا أنها ت